# Cleaning Script

In [192]:
# Import Libraries
import pandas as pd
import numpy as np

# See all columns
pd.set_option('display.max_columns', None)

## Eagles Data

In [193]:
# Import Raw Data
## Press "Choose Files" and select "Eagles Schedule & Scores - Eagles.csv"
from google.colab import files
files.upload()

rawNFLData = pd.read_csv("Eagles Schedule & Scores - Eagles.csv", header = 1)
print(rawNFLData.columns)

Saving Eagles Schedule & Scores - Eagles.csv to Eagles Schedule & Scores - Eagles (2).csv
Index(['Week', 'Day', 'Date', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'OT',
       'Rec', 'Unnamed: 8', 'Opp', 'Tm', 'Opp.1', '1stD', 'TotYd', 'PassY',
       'RushY', 'TO', '1stD.1', 'TotYd.1', 'PassY.1', 'RushY.1', 'TO.1',
       'Offense', 'Defense', 'Sp. Tms', 'Year', 'Season'],
      dtype='object')


In [194]:
# Revising Column Names
NFLData = rawNFLData.drop(columns = ['Day', 'Unnamed: 4', 'TotYd', 'PassY',
                                     'RushY', 'TotYd.1', 'PassY.1', 'RushY.1'])
NFLData = NFLData.rename(columns={'Week': 'Game', 'Date': 'MonthDay',
                                  'Unnamed: 3': 'Time', 'Unnamed: 5': 'Result',
                                  'OT': 'Overtime', 'Unnamed: 8': 'H/A',
                                  'Tm': 'TeamScore', 'Opp.1': 'OppScore',
                                  '1stD': 'Off1D', '1stD.1': 'Def1D',
                                  'TO': 'OffTO', 'TO.1': 'DefTO',
                                  'Offense': 'OffEP', 'Defense': 'DefEP',
                                  'Sp. Tms': 'SpTmEP'})
print(NFLData.columns)
NFLData.head()

Index(['Game', 'MonthDay', 'Time', 'Result', 'Overtime', 'Rec', 'H/A', 'Opp',
       'TeamScore', 'OppScore', 'Off1D', 'OffTO', 'Def1D', 'DefTO', 'OffEP',
       'DefEP', 'SpTmEP', 'Year', 'Season'],
      dtype='object')


,Game,MonthDay,Time,Result,Overtime,Rec,H/A,Opp,TeamScore,OppScore,Off1D,OffTO,Def1D,DefTO,OffEP,DefEP,SpTmEP,Year,Season
0,1,September 7,1:02PM ET,W,NaN,1-0,NaN,Jacksonville Jaguars,34.0,17.0,24.0,3.0,18.0,1.0,-2.13,12.73,3.49,2014,2014
1,2,September 15,8:31PM ET,W,NaN,2-0,@,Indianapolis Colts,30.0,27.0,24.0,1.0,25.0,2.0,18.41,-6.06,-8.50,2014,2014
2,3,September 21,1:02PM ET,W,NaN,3-0,NaN,Washington Redskins,37.0,34.0,22.0,1.0,27.0,1.0,10.89,-18.48,10.95,2014,2014
3,4,September 28,4:25PM ET,L,NaN,3-1,@,San Francisco 49ers,21.0,26.0,11.0,4.0,20.0,1.0,-19.02,5.23,6.62,2014,2014
4,5,October 5,1:02PM ET,W,NaN,4-1,NaN,St. Louis Rams,34.0,28.0,22.0,3.0,29.0,3.0,1.74,-2.19,7.68,2014,2014


In [195]:
# Dealing with NA's
## Droping rows where the week is NaN (spacers)
NFLData = NFLData.dropna(subset=['Game'])

## Dropping bye weeks
NFLData = NFLData[NFLData['Opp'] != 'Bye Week']

In [196]:
# Creating/reformatting new columns
## Date
NFLData['Date'] = NFLData['MonthDay'] + ', ' + NFLData['Year'].astype(str)
NFLData['Date'] = pd.to_datetime(NFLData['Date'])

### Removing games outside of our date range
NFLData['Date'] = pd.to_datetime(NFLData['Date'])
NFLData = NFLData[NFLData['Date'] < '2018-04-30']

## Day of the Week
NFLData['Day'] = NFLData['Date'].dt.day_name()

## Start Time
timeStr = NFLData["Time"].str.split().str[0]
NFLData['StartTime'] = pd.to_datetime(timeStr, format='%I:%M%p',
                                      errors='coerce').dt.strftime('%H:%M:%S')

## End Time
NFLData['StartTime'] = pd.to_datetime(NFLData['StartTime'], format = '%H:%M:%S')
NFLData['xEndTime'] = (NFLData['StartTime'] +
                       pd.Timedelta(hours = 3, minutes = 12)).dt.time
NFLData['StartTime'] = pd.to_datetime(timeStr, format='%I:%M%p',
                                      errors='coerce').dt.strftime('%H:%M:%S')

## Home Games
NFLData['Home'] = np.where(NFLData['H/A'] == '@', 1, 0)

## Point Differential
NFLData['PointDiff'] = NFLData['TeamScore'] - NFLData['OppScore']

## Turnover Differential
NFLData['DefTO'] = pd.to_numeric(NFLData['DefTO'], errors='coerce').fillna(0)
NFLData['OffTO'] = pd.to_numeric(NFLData['OffTO'], errors='coerce').fillna(0)
NFLData['TODiff'] = NFLData['DefTO'] - NFLData['OffTO']

## Result
NFLData['Win'] = np.where(NFLData['Result'] == 'W', 1, 0)

## Overtime
NFLData['OT'] = np.where(NFLData['Overtime'] == 'OT', 1, 0)

## Win/Loss Streak
NFLData[['Wins', 'Losses']] = (NFLData['Rec'].str.split('-',expand = True)
.astype(int))

NFLData['WinChange'] = NFLData['Wins'].diff()
NFLData['LossChange'] = NFLData['Losses'].diff()

NFLData['Outcome'] = np.where(NFLData['WinChange'] == 1, 'W',
                              np.where(NFLData['LossChange'] == 1, 'L', pd.NA))
NFLData.loc[NFLData.index[0], 'Outcome'] = ('W' if NFLData.loc[NFLData.index[0],
                                                               'Wins'] == 1
                                            else 'L')

NFLData['StreakGroup'] = (NFLData['Outcome'] != NFLData['Outcome'].shift()).cumsum()
NFLData['StreakCount'] = NFLData.groupby('StreakGroup').cumcount() + 1
NFLData['Streak'] = NFLData['Streak'] = np.where(NFLData['Outcome'] == 'W',
                                                 NFLData['StreakCount'],
                                                 -NFLData['StreakCount'])

## Rivalry Games
NFLData['Rivalry'] = np.where((NFLData['Opp'] == 'New York Giants') |
 (NFLData['Opp'] == 'Dallas Cowboys'), 1, 0)
NFLData['Giants'] = np.where(NFLData['Opp'] == 'New York Giants', 1, 0)
NFLData['Cowboys'] = np.where(NFLData['Opp'] == 'Dallas Cowboys', 1, 0)

## Game Type
NFLData['Game'] = NFLData['Game'].astype(str)
NFLData['GameType'] = np.where(NFLData['Game'].str.isdigit(),
                            'Regular Season', NFLData['Game'])
GameTypeDummies = pd.get_dummies(NFLData['GameType'], dtype=int)
GameTypeDummies.columns = (GameTypeDummies.columns.str.replace(' ', '')
.str.replace('.', '', regex=False))
NFLData = pd.concat([NFLData, GameTypeDummies], axis = 1)

## Round Stats
floatCols = ['TeamScore', 'OppScore', 'PointDiff', 'DefTO', 'OffTO', 'TODiff']
NFLData[floatCols] = NFLData[floatCols].astype('Int64')

In [197]:
# Keep Desired Columns
NFLData = NFLData[['Game', 'Day', 'Date', 'StartTime', 'xEndTime', 'Home',
                   'Opp', 'TeamScore', 'OppScore', 'PointDiff', 'DefTO',
                   'OffTO', 'TODiff', 'Win', 'OT', 'Season', 'Wins', 'Losses',
                   'Streak', 'Rivalry', 'Giants', 'Cowboys', 'RegularSeason',
                   'Division', 'ConfChamp', 'SuperBowl']]

# Save Cleaned Data
NFLData.to_csv("cleanedEaglesData.csv", index = False)
NFLData

,Game,Day,Date,StartTime,xEndTime,Home,Opp,TeamScore,OppScore,PointDiff,DefTO,OffTO,TODiff,Win,OT,Season,Wins,Losses,Streak,Rivalry,Giants,Cowboys,RegularSeason,Division,ConfChamp,SuperBowl
0,1,Sunday,2014-09-07,13:02:00,16:14:00,0,Jacksonville Jaguars,34,17,17,1,3,-2,1,0,2014,1,0,1,0,0,0,1,0,0,0
1,2,Monday,2014-09-15,20:31:00,23:43:00,1,Indianapolis Colts,30,27,3,2,1,1,1,0,2014,2,0,2,0,0,0,1,0,0,0
2,3,Sunday,2014-09-21,13:02:00,16:14:00,0,Washington Redskins,37,34,3,1,1,0,1,0,2014,3,0,3,0,0,0,1,0,0,0
3,4,Sunday,2014-09-28,16:25:00,19:37:00,1,San Francisco 49ers,21,26,-5,1,4,-3,0,0,2014,3,1,-1,0,0,0,1,0,0,0
4,5,Sunday,2014-10-05,13:02:00,16:14:00,0,St. Louis Rams,34,28,6,3,3,0,1,0,2014,4,1,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,16,Monday,2017-12-25,20:30:00,23:42:00,0,Oakland Raiders,19,10,9,5,2,3,1,0,2017,13,2,3,0,0,0,1,0,0,0
67,17,Sunday,2017-12-31,13:00:00,16:12:00,0,Dallas Cowboys,0,6,-6,1,2,-1,0,0,2017,13,3,-1,1,0,1,1,0,0,0
69,Division,Saturday,2018-01-13,16:35:00,19:47:00,0,Atlanta Falcons,15,10,5,0,2,-2,1,0,2017,14,3,1,0,0,0,0,1,0,0
70,Conf. Champ.,Sunday,2018-01-21,18:40:00,21:52:00,0,Minnesota Vikings,38,7,31,3,0,3,1,0,2017,15,3,2,0,0,0,0,0,1,0


## Flyers Data

In [198]:
# Import Raw Data
## Press "Choose Files" and select "Flyers.csv"
from google.colab import files
files.upload()

rawNHLdata = pd.read_csv("Flyers.csv", header = 0)
print(rawNHLdata.columns)

Saving Flyers.csv to Flyers (2).csv
Index(['GP', 'Date', 'HomeAway', 'Opponent', 'GF', 'GA', 'WinLoss', 'OTSO',
       'W', 'L', 'OL', 'Streak', 'Attend.', 'LOG'],
      dtype='object')


In [199]:
# Revising Column Names
NHLData = rawNHLdata.drop(columns = ['OL'])
NHLData = NHLData.rename(columns={'GP': 'Game', 'HomeAway': 'H/A',
                                  'Opponent': 'Opp', 'GF': 'TeamScore',
                                  'GA': 'OppScore', 'WinLoss': 'Result',
                                  'W': 'Wins', 'L': 'Losses',
                                  'Attend.': 'Attendance'})
print(NHLData.columns)
NHLData.head()

Index(['Game', 'Date', 'H/A', 'Opp', 'TeamScore', 'OppScore', 'Result', 'OTSO',
       'Wins', 'Losses', 'Streak', 'Attendance', 'LOG'],
      dtype='object')


/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Game,Date,H/A,Opp,TeamScore,OppScore,Result,OTSO,Wins,Losses,Streak,Attendance,LOG
0,41,1/2/2014,@,Colorado Avalanche,1,2,L,NaN,20,17,L 1,NaN,NaN
1,42,1/4/2014,@,Phoenix Coyotes,5,3,W,NaN,21,17,W 1,NaN,NaN
2,43,1/7/2014,@,New Jersey Devils,3,2,W,OT,22,17,W 2,NaN,NaN
3,44,1/8/2014,NaN,Montreal Canadiens,3,1,W,NaN,23,17,W 3,NaN,NaN
4,45,1/11/2014,NaN,Tampa Bay Lightning,3,6,L,NaN,23,18,L 1,NaN,NaN


In [200]:
# Creating/reformatting new columns
## Date + Year
NHLData['Date'] = pd.to_datetime(NHLData['Date'])

## Day of the Week
NHLData['Day'] = NHLData['Date'].dt.day_name()

## Start Time
NHLData['xStartTime'] = np.where(NHLData['Date'].dt.dayofweek >= 5,
                                    NHLData['Date'] + pd.Timedelta(hours = 17),
                                    NHLData['Date'] + pd.Timedelta(hours = 19))

## End Time
NHLData['LOG'] = NHLData['LOG'].astype(str)

time_split = NHLData['LOG'].str.split(':', expand=True)
hours = pd.to_numeric(time_split[0], errors='coerce')
minutes = pd.to_numeric(time_split[1], errors='coerce')

NHLData['LOGMins'] = hours * 60 + minutes
NHLData['xLOG'] = NHLData['LOGMins'].fillna(150).astype(int)

NHLData['xEndTime'] = NHLData['xStartTime'] + pd.to_timedelta(
    NHLData['xLOG'], unit = 'm')

NHLData['xStartTime'] = NHLData['xStartTime'].dt.time
NHLData['xEndTime'] = NHLData['xEndTime'].dt.time

## Home
NHLData['Home'] = np.where(NHLData['H/A'] == '@', 1, 0)

## Attendance
NHLData['Attendance'] = pd.to_numeric(NHLData['Attendance'],
                                      errors='coerce').astype('Int64')

## Point Differential
NHLData['PointDiff'] = NHLData['TeamScore'] - NHLData['OppScore']

## Result
NHLData['Win'] = np.where(NHLData['Result'] == 'W', 1, 0)

## Overtime
NHLData['OT'] = np.where((NHLData['OTSO'] == 'OT') |
                            (NHLData['OTSO'] == 'SO'), 1, 0)

## Shootout
NHLData['SO'] = np.where(NHLData['OTSO'] == 'SO', 1, 0)

## Season
NHLData['Season'] = np.where(NHLData['Date'].dt.month >= 10,
                             NHLData['Date'].dt.year,
                             NHLData['Date'].dt.year - 1)

## Win/Loss Streak
NHLData['WinChange'] = NHLData['Wins'].diff()
NHLData['LossChange'] = NHLData['Losses'].diff()

NHLData['Outcome'] = np.where(NHLData['WinChange'] == 1, 'W',
                           np.where(NHLData['LossChange'] == 1, 'L', pd.NA))
NHLData.loc[NHLData.index[0], 'Outcome'] = 'L'

NHLData['StreakGroup'] = (NHLData['Outcome'] !=
                             NHLData['Outcome'].shift()).cumsum()
NHLData['StreakCount'] = NHLData.groupby('StreakGroup').cumcount() + 1
NHLData['Streak'] = NHLData['Streak'] = np.where(NHLData['Outcome'] == 'W',
                                                       NHLData['StreakCount'],
                                                       -NHLData['StreakCount'])

## Rivalry Games
NHLData['Rivalry'] = np.where((NHLData['Opp'] == 'Pittsburgh Penguins') |
 (NHLData['Opp'] == 'New York Rangers') |
  (NHLData['Opp'] == 'New Jersey Devils'), 1, 0)
NHLData['Penguins'] = np.where(NHLData['Opp'] == 'Pittsburgh Penguins', 1, 0)
NHLData['Rangers'] = np.where(NHLData['Opp'] == 'New York Rangers', 1, 0)
NHLData['Devils'] = np.where(NHLData['Opp'] == 'New Jersey Devils', 1, 0)

## Game Type
flyersConfQF = pd.to_datetime(['2014-04-17', '2014-04-20', '2014-04-22',
                               '2014-04-25', '2014-04-27', '2014-04-29',
                               '2014-04-30', '2016-04-14', '2016-04-16',
                               '2016-04-18', '2016-04-20', '2016-04-22',
                               '2016-04-24', '2018-04-11', '2018-04-13',
                               '2018-04-15', '2018-04-18', '2018-04-20',
                               '2018-04-22'])
NHLData['RegularSeason'] = np.where(NHLData['Date'].isin(flyersConfQF),
                                       0, 1)
NHLData['ConfQF'] = np.where(NHLData['Date'].isin(flyersConfQF), 1, 0)

In [201]:
# Keep Desired Columns
NHLData = NHLData[['Game', 'Day', 'Date', 'xStartTime', 'xLOG', 'xEndTime',
                   'Home', 'Attendance', 'Opp', 'TeamScore', 'OppScore',
                   'PointDiff', 'Win', 'OT', 'SO', 'Season', 'Wins', 'Losses',
                   'Streak', 'Rivalry', 'Penguins', 'Rangers', 'Devils',
                   'RegularSeason', 'ConfQF']]

# Save Cleaned Data
NHLData.to_csv("cleanedFlyersData.csv", index = False)

NHLData

,Game,Day,Date,xStartTime,xLOG,xEndTime,Home,Attendance,Opp,TeamScore,OppScore,PointDiff,Win,OT,SO,Season,Wins,Losses,Streak,Rivalry,Penguins,Rangers,Devils,RegularSeason,ConfQF
0,41,Thursday,2014-01-02,19:00:00,150,21:30:00,1,<NA>,Colorado Avalanche,1,2,-1,0,0,0,2013,20,17,-1,0,0,0,0,1,0
1,42,Saturday,2014-01-04,17:00:00,150,19:30:00,1,<NA>,Phoenix Coyotes,5,3,2,1,0,0,2013,21,17,1,0,0,0,0,1,0
2,43,Tuesday,2014-01-07,19:00:00,150,21:30:00,1,<NA>,New Jersey Devils,3,2,1,1,1,0,2013,22,17,2,1,0,0,1,1,0
3,44,Wednesday,2014-01-08,19:00:00,150,21:30:00,0,<NA>,Montreal Canadiens,3,1,2,1,0,0,2013,23,17,3,0,0,0,0,1,0
4,45,Saturday,2014-01-11,17:00:00,150,19:30:00,0,<NA>,Tampa Bay Lightning,3,6,-3,0,0,0,2013,23,18,-1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384,2,Friday,2018-04-13,19:00:00,155,21:35:00,1,18648,Pittsburgh Penguins,5,1,4,1,0,0,2017,1,1,1,1,1,0,0,0,1
385,3,Sunday,2018-04-15,17:00:00,152,19:32:00,0,19955,Pittsburgh Penguins,1,5,-4,0,0,0,2017,1,2,-1,1,1,0,0,0,1
386,4,Wednesday,2018-04-18,19:00:00,146,21:26:00,0,19644,Pittsburgh Penguins,0,5,-5,0,0,0,2017,1,3,-2,1,1,0,0,0,1
387,5,Friday,2018-04-20,19:00:00,151,21:31:00,1,18632,Pittsburgh Penguins,4,2,2,1,0,0,2017,2,3,1,1,1,0,0,0,1


## 76ers Data

In [202]:
# Import Raw Data
## Press "Choose Files" and select "Philly76.csv"
from google.colab import files
files.upload()

# Import Raw Data
rawNBAData = pd.read_csv("Philly76.csv", header = 0)
print(rawNBAData.columns)

Saving Philly76.csv to Philly76 (3).csv
Index(['Game', 'Date', 'Start (ET)', 'HomeAway', 'Opponent', 'WinLoss', 'OT',
       'Tm', 'Opp', 'W', 'L', 'Streak', 'Attend.', 'LOG'],
      dtype='object')


In [203]:
# Revising Column Names
NBAData = rawNBAData.rename(columns={'Start (ET)': 'StartTime',
                                     'HomeAway': 'H/A', 'Tm': 'TeamScore',
                                     'Opp': 'OppScore', 'Opponent': 'Opp',
                                     'WinLoss': 'Result', 'W': 'Wins',
                                     'L': 'Losses', 'Attend.': 'Attendance'})
print(NBAData.columns)
NBAData.head()

Index(['Game', 'Date', 'StartTime', 'H/A', 'Opp', 'Result', 'OT', 'TeamScore',
       'OppScore', 'Wins', 'Losses', 'Streak', 'Attendance', 'LOG'],
      dtype='object')


,Game,Date,StartTime,H/A,Opp,Result,OT,TeamScore,OppScore,Wins,Losses,Streak,Attendance,LOG
0,31,1/1/2014,9:00p,@,Denver Nuggets,W,NaN,114,102,10,21,W 2,16006,2:14:00 AM
1,32,1/2/2014,10:00p,@,Sacramento Kings,W,NaN,113,104,11,21,W 3,16259,2:23:00 AM
2,33,1/4/2014,10:00p,@,Portland Trail Blazers,W,NaN,101,99,12,21,W 4,20004,2:25:00 AM
3,34,1/6/2014,7:00p,NaN,Minnesota Timberwolves,L,NaN,95,126,12,22,L 1,10736,2:01:00 AM
4,35,1/7/2014,7:00p,@,Cleveland Cavaliers,L,NaN,93,111,12,23,L 2,13344,2:07:00 AM


In [204]:
# Creating/reformatting new columns
## Date + Year
NBAData['Date'] = pd.to_datetime(NBAData['Date'])

## Day of the Week
NBAData['Day'] = NBAData['Date'].dt.day_name()

## Start Time
NBAData['StartTime'] = (NBAData['StartTime']
                        .str.replace('p', ' PM', regex = False)
                        .str.replace('a', ' AM', regex = False))
NBAData['StartTime'] = pd.to_datetime(NBAData['StartTime'],
                                      format='%I:%M %p')

## End Time
NBAData['LOG'] = pd.to_timedelta(NBAData['LOG'])
NBAData['LOG'] = (NBAData['LOG'].dt.total_seconds() / 60).astype(int)

NBAData['EndTime'] = NBAData['StartTime'] + pd.to_timedelta(
    NBAData['LOG'], unit = 'm')

NBAData['StartTime'] = NBAData['StartTime'].dt.time
NBAData['EndTime'] = NBAData['EndTime'].dt.time

## Home
NBAData['Home'] = np.where(NBAData['H/A'] == '@', 1, 0)

## Point Differential
NBAData['PointDiff'] = NBAData['TeamScore'] - NBAData['OppScore']

## Result
NBAData['Win'] = np.where(NBAData['Result'] == 'W', 1, 0)

## Overtime
NBAData['OT'] = np.where(NBAData['OT'] == 'OT', 1, 0)

## Season
NBAData['Season'] = np.where(NBAData['Date'].dt.month >= 10,
                             NBAData['Date'].dt.year,
                             NBAData['Date'].dt.year - 1)

## Win/Loss Streak
NBAData['WinChange'] = NBAData['Wins'].diff()
NBAData['LossChange'] = NBAData['Losses'].diff()

NBAData['Outcome'] = np.where(NBAData['WinChange'] == 1, 'W',
                              np.where(NBAData['LossChange'] == 1, 'L', pd.NA))
NBAData.loc[NBAData.index[0], 'Outcome'] = 'W'

NBAData['StreakGroup'] = (NBAData['Outcome'] !=
                          NBAData['Outcome'].shift()).cumsum()
NBAData['StreakCount'] = NBAData.groupby('StreakGroup').cumcount() + 1
NBAData['Streak'] = NBAData['Streak'] = np.where(NBAData['Outcome'] == 'W',
                                                 NBAData['StreakCount'],
                                                 -NBAData['StreakCount'])

## Rivalry Games
NBAData['Rivalry'] = np.where((NBAData['Opp'] == 'Boston Celtics') |
                              (NBAData['Opp'] == 'New York Knicks') , 1, 0)
NBAData['Celtics'] = np.where(NBAData['Opp'] == 'Boston Celtics', 1, 0)
NBAData['Knicks'] = np.where(NBAData['Opp'] == 'New York Knicks', 1, 0)

In [205]:
# Keep Desired Columns
NBAData = NBAData[['Game', 'Day', 'Date', 'StartTime', 'LOG', 'EndTime', 'Home',
                   'Attendance', 'Opp', 'TeamScore', 'OppScore', 'PointDiff',
                   'Win', 'OT', 'Season', 'Wins', 'Losses', 'Streak', 'Rivalry',
                   'Celtics', 'Knicks']]

# Save Cleaned Data
NBAData.to_csv("cleaned76ersData.csv", index = False)

NBAData

,Game,Day,Date,StartTime,LOG,EndTime,Home,Attendance,Opp,TeamScore,OppScore,PointDiff,Win,OT,Season,Wins,Losses,Streak,Rivalry,Celtics,Knicks
0,31,Wednesday,2014-01-01,21:00:00,134,23:14:00,1,16006,Denver Nuggets,114,102,12,1,0,2013,10,21,1,0,0,0
1,32,Thursday,2014-01-02,22:00:00,143,00:23:00,1,16259,Sacramento Kings,113,104,9,1,0,2013,11,21,2,0,0,0
2,33,Saturday,2014-01-04,22:00:00,145,00:25:00,1,20004,Portland Trail Blazers,101,99,2,1,0,2013,12,21,3,0,0,0
3,34,Monday,2014-01-06,19:00:00,121,21:01:00,0,10736,Minnesota Timberwolves,95,126,-31,0,0,2013,12,22,-1,0,0,0
4,35,Tuesday,2014-01-07,19:00:00,127,21:07:00,1,13344,Cleveland Cavaliers,93,111,-18,0,0,2013,12,23,-2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,78,Wednesday,2018-04-04,19:00:00,131,21:11:00,1,18395,Detroit Pistons,115,108,7,1,0,2017,48,30,12,0,0,0
376,79,Friday,2018-04-06,19:00:00,146,21:26:00,0,20769,Cleveland Cavaliers,132,130,2,1,0,2017,49,30,13,0,0,0
377,80,Sunday,2018-04-08,13:00:00,122,15:02:00,0,20846,Dallas Mavericks,109,97,12,1,0,2017,50,30,14,0,0,0
378,81,Tuesday,2018-04-10,19:30:00,140,21:50:00,1,15673,Atlanta Hawks,121,113,8,1,0,2017,51,30,15,0,0,0


## Phillies

In [206]:
# Import Raw Data
## Press "Choose Files" and select "Phillies_Schedule_Updated_2.csv"
from google.colab import files
files.upload()

# Import Raw Data
rawMLBData = pd.read_csv("Phillies_Schedule_Updated_2.csv", header = 0)
print(rawMLBData.columns)

Saving Phillies_Schedule_Updated_2.csv to Phillies_Schedule_Updated_2 (1).csv
Index(['Game', 'Date', 'Tm', 'Opp', 'W.L', 'R', 'RA', 'Rank_Standings',
       'Game_Length', 'Day_Night', 'Attendance', 'CLI', 'Streak', 'Home',
       'FullDate', 'Year', 'start_time_ET'],
      dtype='object')


In [207]:
# Revising Column Names
MLBData = rawMLBData.drop(columns = ['Tm', 'Year', 'FullDate'])
MLBData = MLBData.rename(columns = {'R': 'TeamScore', 'RA': 'OppScore',
                                    'W.L': 'Result', 'Rank_Standings': 'Rank',
                                    'Game_Length': 'LOG',
                                    'start_time_ET': 'StartTime'})
print(MLBData.columns)
MLBData.head()

Index(['Game', 'Date', 'Opp', 'Result', 'TeamScore', 'OppScore', 'Rank', 'LOG',
       'Day_Night', 'Attendance', 'CLI', 'Streak', 'Home', 'StartTime'],
      dtype='object')


,Game,Date,Opp,Result,TeamScore,OppScore,Rank,LOG,Day_Night,Attendance,CLI,Streak,Home,StartTime
0,1,3/31/2014,TEX,W,14,10,1,3:36,D,49031.0,0.96,+,1,2:05 PM
1,2,4/1/2014,TEX,L-wo,2,3,3,3:14,N,29530.0,0.98,-,1,8:05 PM
2,3,4/2/2014,TEX,L-wo,3,4,4,3:02,N,28218.0,0.88,--,1,8:05 PM
3,4,4/4/2014,CHC,W,7,2,4,3:16,D,38283.0,0.88,+,1,2:20 PM
4,5,4/5/2014,CHC,W,2,0,3,2:53,D,30651.0,0.92,++,1,2:20 PM


In [208]:
# Creating/reformatting new columns
## Date + Year
MLBData['Date'] = pd.to_datetime(MLBData['Date'])

## Day of the Week
MLBData['Day'] = MLBData['Date'].dt.day_name()

## Result
MLBData['Win'] = MLBData['Result'].str.contains('W').astype(int)

## Walkoff
MLBData['Walkoff'] = MLBData['Result'].str.contains('wo').astype(int)

## Point Differential
MLBData['PointDiff'] = MLBData['TeamScore'] - MLBData['OppScore']

## Day Game
MLBData['DayGame'] = MLBData['Day_Night'].str.contains('D').astype(int)

## Start Time
MLBData['StartTime'] = pd.to_datetime(MLBData['StartTime'],
                                      format='%I:%M %p')

## End Time
MLBData['LOG'] = MLBData['LOG'].astype(str)

time_split = MLBData['LOG'].str.split(':', expand=True)
hours = pd.to_numeric(time_split[0], errors='coerce')
minutes = pd.to_numeric(time_split[1], errors='coerce')

MLBData['LOG'] = hours * 60 + minutes

MLBData['EndTime'] = MLBData['StartTime'] + pd.to_timedelta(
    MLBData['LOG'], unit = 'm')

MLBData['StartTime'] = MLBData['StartTime'].dt.time
MLBData['EndTime'] = MLBData['EndTime'].dt.time

## Attendance
MLBData['Attendance'] = pd.to_numeric(MLBData['Attendance'],
                                      errors='coerce').astype('Int64')

## Season
MLBData['Season'] = np.where(MLBData['Date'].dt.month >= 3,
                             MLBData['Date'].dt.year,
                             MLBData['Date'].dt.year - 1)

## Streak
MLBData['Streak'] = (MLBData['Streak'].str.len() *
                     MLBData['Streak'].str.contains('-', na=False)
                     .map({True: -1, False: 1}))

## Rivalry Games
MLBData['Rivalry'] = np.where((MLBData['Opp'] == 'NYM') |
                              (MLBData['Opp'] == 'ATL') , 1, 0)
MLBData['Mets'] = np.where(MLBData['Opp'] == 'NYM', 1, 0)
MLBData['Braves'] = np.where(MLBData['Opp'] == 'ATL', 1, 0)

In [209]:
# Keep Desired Columns
MLBData = MLBData[['Game', 'Day', 'Date', 'StartTime', 'LOG', 'EndTime',
                   'DayGame', 'Rank', 'CLI', 'Home', 'Attendance', 'Opp',
                   'TeamScore', 'OppScore', 'PointDiff', 'Win', 'Walkoff',
                   'Season', 'Streak', 'Rivalry', 'Mets', 'Braves']]

# Save Cleaned Data
MLBData.to_csv("cleanedPhilliesData.csv", index = False)

MLBData

,Game,Day,Date,StartTime,LOG,EndTime,DayGame,Rank,CLI,Home,Attendance,Opp,TeamScore,OppScore,PointDiff,Win,Walkoff,Season,Streak,Rivalry,Mets,Braves
0,1,Monday,2014-03-31,14:05:00,216,17:41:00,1,1,0.96,1,49031,TEX,14,10,4,1,0,2014,1,0,0,0
1,2,Tuesday,2014-04-01,20:05:00,194,23:19:00,0,3,0.98,1,29530,TEX,2,3,-1,0,1,2014,-1,0,0,0
2,3,Wednesday,2014-04-02,20:05:00,182,23:07:00,0,4,0.88,1,28218,TEX,3,4,-1,0,1,2014,-2,0,0,0
3,4,Friday,2014-04-04,14:20:00,196,17:36:00,1,4,0.88,1,38283,CHC,7,2,5,1,0,2014,1,0,0,0
4,5,Saturday,2014-04-05,14:20:00,173,17:13:00,1,3,0.92,1,30651,CHC,2,0,2,1,0,2014,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
806,158,Wednesday,2018-09-26,20:40:00,171,23:31:00,0,3,0.00,1,35181,COL,0,14,-14,0,0,2018,-7,0,0,0
807,159,Thursday,2018-09-27,15:10:00,186,18:16:00,1,3,0.00,1,36448,COL,3,5,-2,0,0,2018,-8,0,0,0
808,160,Friday,2018-09-28,19:05:00,222,22:47:00,0,3,0.00,0,24306,ATL,2,10,-8,0,0,2018,-9,1,0,1
809,161,Saturday,2018-09-29,19:05:00,163,21:48:00,0,3,0.00,0,30886,ATL,3,0,3,1,0,2018,1,1,0,1
